# Jane Street 模型训练 - XGBoost

## 为什么选择XGBoost？

### XGBoost的优势

**1. 稳定性和可靠性**
- 成熟稳定的算法，在生产环境广泛使用
- 对异常值和噪声具有较强的鲁棒性
- 内置正则化防止过拟合

**2. 性能优异**
- 在表格数据上通常表现最佳
- 支持并行处理和GPU加速
- 内存效率高，适合大规模数据

**3. 功能丰富**
- 支持自定义损失函数和评估指标
- 丰富的参数可供调优
- 处理缺失值能力强
- 支持单调性约束

### LightGBM vs XGBoost

| 特性 | LightGBM | XGBoost |
|------|----------|----------|
| 训练速度 | 更快 | 快 |
| 内存占用 | 更低 | 低 |
| 稳定性 | 好 | 更好 |
| 调参难度 | 中等 | 中等 |
| 社区支持 | 活跃 | 非常活跃 |
| 生产使用 | 广泛 | 非常广泛 |

**建议策略：** 两者都尝试，然后选择表现更好的，或者将两者集成。

## 1. 环境设置

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import warnings
import joblib
import gc
from tqdm import tqdm

import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# 设置路径
# 使用相对路径
DATA_PATH = Path('../Dataset')
PROCESSED_PATH = Path('./processed_data')
MODEL_OUTPUT_PATH = Path('./models')

# 创建模型输出目录
MODEL_OUTPUT_PATH.mkdir(exist_ok=True, parents=True)

print(f"数据路径: {DATA_PATH}")
print(f"处理后的数据路径: {PROCESSED_PATH}")
print(f"模型输出路径: {MODEL_OUTPUT_PATH}")

## 2. 配置参数

In [ ]:
class CONFIG:
    # 数据路径
    TRAIN_FILE = PROCESSED_PATH / 'training.parquet'
    VALID_FILE = PROCESSED_PATH / 'validation.parquet'
    
    # 特征列
    FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{i}_lag_1" for i in range(9)]
    
    # 目标列
    TARGET_COL = "responder_6"
    
    # 权重列
    WEIGHT_COL = "weight"
    
    # 训练参数
    SEED = 42
    N_FOLDS = 5
    N_ESTIMATORS = 2000
    EARLY_STOPPING_ROUNDS = 100
    
    # XGBoost参数
    LEARNING_RATE = 0.05
    MAX_DEPTH = 6
    MIN_CHILD_WEIGHT = 100
    SUBSAMPLE = 0.8
    COLSAMPLE_BYTREE = 0.8
    REG_ALPHA = 1
    REG_LAMBDA = 1
    
    # 系统参数
    USE_GPU = True  # 如果没有GPU，设置为False
    TREE_METHOD = 'hist'  # 'hist'或'gpu_hist'
    
print("=== 训练配置 ===")
print(f"特征数量: {len(CONFIG.FEATURE_COLS)}")
print(f"目标列: {CONFIG.TARGET_COL}")
print(f"交叉验证折数: {CONFIG.N_FOLDS}")
print(f"使用GPU: {CONFIG.USE_GPU}")

## 3. 自定义评估指标

### XGBoost评估指标格式

XGBoost的自定义评估指标需要特定的签名：
- 接受参数：`preds, dmatrix`
- 返回格式：`metric_name, metric_value`
- 注意：XGBoost最大化指标，所以如果我们要最大化R²，需要返回负的R²

In [ ]:
def weighted_r2_score(y_true, y_pred, sample_weight):
    """
    计算加权R²得分
    
    公式: R² = 1 - Σ(wi * (yi - ŷi)²) / Σ(wi * yi²)
    """
    numerator = np.average((y_true - y_pred) ** 2, weights=sample_weight)
    denominator = np.average(y_true ** 2, weights=sample_weight) + 1e-38
    r2 = 1 - numerator / denominator
    return r2

# XGBoost评估函数
def xgb_r2_metric(preds, dmatrix):
    """
    XGBoost格式的加权R²评估函数
    
    注意：XGBoost最大化指标，而我们想要最大化R²
    所以直接返回R²（因为R²越高越好）
    """
    y_true = dmatrix.get_label()
    sample_weight = dmatrix.get_weight()
    
    r2 = weighted_r2_score(y_true, preds, sample_weight)
    
    # 返回格式：名称, 值
    # XGBoost会最大化这个值
    return 'r2', r2

# sklearn格式的评估函数
def evaluate_model(model, X, y, w):
    """
    评估模型性能
    """
    y_pred = model.predict(X)
    r2 = weighted_r2_score(y, y_pred, w)
    return r2

print("评估函数已定义")

## 4. 数据加载和预处理

In [ ]:
def load_data():
    """
    加载训练和验证数据
    """
    
    if not CONFIG.TRAIN_FILE.exists():
        print("警告: 处理后的训练数据不存在!")
        print("请先运行滞后特征工程笔记本创建数据。")
        return None, None
    
    required_cols = CONFIG.FEATURE_COLS + [CONFIG.TARGET_COL, CONFIG.WEIGHT_COL, 'date_id', 'symbol_id']
    
    print("加载训练数据...")
    train_pl = pl.read_parquet(CONFIG.TRAIN_FILE)
    available_cols = [col for col in required_cols if col in train_pl.columns]
    train_pl = train_pl.select(available_cols)
    train_df = train_pl.to_pandas()
    
    # 处理缺失值
    for col in CONFIG.FEATURE_COLS:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
    
    print(f"训练数据形状: {train_df.shape}")
    print(f"日期范围: {train_df['date_id'].min()} 到 {train_df['date_id'].max()}")
    
    if CONFIG.VALID_FILE.exists():
        print("加载验证数据...")
        valid_pl = pl.read_parquet(CONFIG.VALID_FILE)
        available_cols = [col for col in required_cols if col in valid_pl.columns]
        valid_pl = valid_pl.select(available_cols)
        valid_df = valid_pl.to_pandas()
        
        for col in CONFIG.FEATURE_COLS:
            if col in valid_df.columns:
                valid_df[col] = valid_df[col].fillna(0)
        
        print(f"验证数据形状: {valid_df.shape}")
        print(f"日期范围: {valid_df['date_id'].min()} 到 {valid_df['date_id'].max()}")
    else:
        print("验证数据不存在，将使用交叉验证")
        valid_df = None
    
    return train_df, valid_df

train_df, valid_df = load_data()

## 5. XGBoost模型定义

In [ ]:
def create_xgb_model():
    """
    创建XGBoost模型
    
    参数说明:
    - n_estimators: 树的数量
    - learning_rate: 学习率（eta）
    - max_depth: 树的最大深度
    - min_child_weight: 子节点的最小权重和
    - subsample: 每棵树的样本采样比例
    - colsample_bytree: 每棵树的特征采样比例
    - reg_alpha: L1正则化参数
    - reg_lambda: L2正则化参数
    - tree_method: 树构建方法
    """
    params = {
        'n_estimators': CONFIG.N_ESTIMATORS,
        'learning_rate': CONFIG.LEARNING_RATE,
        'max_depth': CONFIG.MAX_DEPTH,
        'min_child_weight': CONFIG.MIN_CHILD_WEIGHT,
        'subsample': CONFIG.SUBSAMPLE,
        'colsample_bytree': CONFIG.COLSAMPLE_BYTREE,
        'reg_alpha': CONFIG.REG_ALPHA,
        'reg_lambda': CONFIG.REG_LAMBDA,
        'objective': 'reg:squarederror',
        'eval_metric': xgb_r2_metric,  # 自定义评估指标
        'random_state': CONFIG.SEED,
        'tree_method': CONFIG.TREE_METHOD,
        'disable_default_eval_metric': True,  # 禁用默认指标
    }
    
    # GPU设置
    if CONFIG.USE_GPU:
        try:
            # 检查GPU是否可用
            params['tree_method'] = 'hist'
            params['device'] = 'cuda'
            print("使用GPU训练")
        except:
            print("GPU不可用，使用CPU训练")
            params['tree_method'] = 'hist'
    
    model = xgb.XGBRegressor(**params)
    return model

print("XGBoost模型创建函数已定义")

## 6. 交叉验证训练

In [ ]:
def train_with_cv(train_df, valid_df=None):
    """
    使用交叉验证训练模型
    """
    
    models = []
    cv_scores = []
    
    if valid_df is not None:
        # 使用固定的训练/验证集
        print("使用固定的训练/验证集")
        
        X_train = train_df[CONFIG.FEATURE_COLS]
        y_train = train_df[CONFIG.TARGET_COL]
        w_train = train_df[CONFIG.WEIGHT_COL]
        
        X_valid = valid_df[CONFIG.FEATURE_COLS]
        y_valid = valid_df[CONFIG.TARGET_COL]
        w_valid = valid_df[CONFIG.WEIGHT_COL]
        
        model = create_xgb_model()
        
        print("\n开始训练...")
        model.fit(
            X_train, y_train, w_train,
            eval_set=[(X_valid, y_valid, w_valid)],
            verbose=50,
            early_stopping_rounds=CONFIG.EARLY_STOPPING_ROUNDS,
        )
        
        train_r2 = evaluate_model(model, X_train, y_train, w_train)
        valid_r2 = evaluate_model(model, X_valid, y_valid, w_valid)
        
        print(f"\n训练集 R²: {train_r2:.6f}")
        print(f"验证集 R²: {valid_r2:.6f}")
        
        models.append(model)
        cv_scores.append(valid_r2)
        
        model_path = MODEL_OUTPUT_PATH / 'xgb_model_single.pkl'
        joblib.dump(model, model_path)
        print(f"\n模型已保存到: {model_path}")
        
    else:
        # 使用时间序列交叉验证
        print(f"使用 {CONFIG.N_FOLDS} 折时间序列交叉验证")
        
        unique_dates = sorted(train_df['date_id'].unique())
        
        for fold in range(CONFIG.N_FOLDS):
            print(f"\n{'='*20} 折 {fold + 1}/{CONFIG.N_FOLDS} {'='*20}")
            
            # 分割日期
            valid_dates = unique_dates[fold::CONFIG.N_FOLDS]
            train_dates = [d for d in unique_dates if d not in valid_dates]
            
            # 分割数据
            train_fold = train_df[train_df['date_id'].isin(train_dates)]
            valid_fold = train_df[train_df['date_id'].isin(valid_dates)]
            
            X_train = train_fold[CONFIG.FEATURE_COLS]
            y_train = train_fold[CONFIG.TARGET_COL]
            w_train = train_fold[CONFIG.WEIGHT_COL]
            
            X_valid = valid_fold[CONFIG.FEATURE_COLS]
            y_valid = valid_fold[CONFIG.TARGET_COL]
            w_valid = valid_fold[CONFIG.WEIGHT_COL]
            
            print(f"训练集大小: {len(X_train)}")
            print(f"验证集大小: {len(X_valid)}")
            
            model = create_xgb_model()
            
            model.fit(
                X_train, y_train, w_train,
                eval_set=[(X_valid, y_valid, w_valid)],
                verbose=50,
                early_stopping_rounds=CONFIG.EARLY_STOPPING_ROUNDS,
            )
            
            train_r2 = evaluate_model(model, X_train, y_train, w_train)
            valid_r2 = evaluate_model(model, X_valid, y_valid, w_valid)
            
            print(f"训练集 R²: {train_r2:.6f}")
            print(f"验证集 R²: {valid_r2:.6f}")
            
            models.append(model)
            cv_scores.append(valid_r2)
            
            model_path = MODEL_OUTPUT_PATH / f'xgb_model_fold{fold}.pkl'
            joblib.dump(model, model_path)
            print(f"模型已保存到: {model_path}")
            
            del train_fold, valid_fold, model
            gc.collect()
    
    print("\n" + "="*50)
    print("训练完成!")
    print(f"平均验证集 R²: {np.mean(cv_scores):.6f} ± {np.std(cv_scores):.6f}")
    print(f"各折得分: {cv_scores}")
    print("="*50)
    
    return models, cv_scores

if train_df is not None:
    models, cv_scores = train_with_cv(train_df, valid_df)

## 7. 特征重要性分析

In [ ]:
def analyze_feature_importance(models, top_n=20):
    """
    分析特征重要性
    """
    
    importance_list = []
    for i, model in enumerate(models):
        imp = model.feature_importances_
        importance_list.append(imp)
    
    avg_importance = np.mean(importance_list, axis=0)
    
    importance_df = pd.DataFrame({
        'feature': CONFIG.FEATURE_COLS,
        'importance': avg_importance
    })
    importance_df = importance_df.sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(top_n), importance_df['importance'].head(top_n)[::-1])
    plt.yticks(range(top_n), importance_df['feature'].head(top_n)[::-1])
    plt.xlabel('平均重要性')
    plt.title(f'XGBoost - 前{top_n}个重要特征')
    plt.tight_layout()
    plt.show()
    
    print(f"\n前{top_n}个重要特征:")
    print(importance_df.head(top_n))
    
    return importance_df

if 'models' in locals() and models:
    importance_df = analyze_feature_importance(models)

## 8. 学习曲线分析

In [ ]:
def plot_learning_curve(model):
    """
    绘制学习曲线
    """
    # 获取评估历史
    results = model.evals_result()
    
    if results:
        plt.figure(figsize=(10, 6))
        
        # 假设只有一个验证集
        eval_key = list(results.keys())[0]
        if 'r2' in results[eval_key]:
            r2_history = results[eval_key]['r2']
            plt.plot(r2_history, label='Validation R²')
            plt.xlabel('Iteration')
            plt.ylabel('R² Score')
            plt.title('XGBoost Learning Curve')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()
        else:
            print("没有R²历史记录")
    else:
        print("模型没有评估历史")

if 'models' in locals() and models:
    plot_learning_curve(models[0])

## 9. 按股票评估

In [ ]:
def evaluate_by_symbol(models, df):
    """
    按股票评估模型性能
    """
    
    eval_df = valid_df if valid_df is not None else train_df
    
    symbol_scores = {}
    
    for symbol_id in sorted(eval_df['symbol_id'].unique()):
        symbol_data = eval_df[eval_df['symbol_id'] == symbol_id]
        
        X = symbol_data[CONFIG.FEATURE_COLS]
        y = symbol_data[CONFIG.TARGET_COL]
        w = symbol_data[CONFIG.WEIGHT_COL]
        
        predictions = np.mean([model.predict(X) for model in models], axis=0)
        r2 = weighted_r2_score(y.values, predictions, w.values)
        
        symbol_scores[symbol_id] = {
            'r2': r2,
            'samples': len(symbol_data)
        }
    
    symbols = list(symbol_scores.keys())
    scores = [symbol_scores[s]['r2'] for s in symbols]
    
    plt.figure(figsize=(15, 5))
    plt.bar(symbols, scores)
    plt.xlabel('股票ID')
    plt.ylabel('R² 得分')
    plt.title('各股票的R²得分')
    plt.axhline(y=np.mean(scores), color='red', linestyle='--', label=f'平均: {np.mean(scores):.4f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return symbol_scores

if 'models' in locals() and models:
    symbol_scores = evaluate_by_symbol(models, train_df)

## 10. 保存训练信息

In [ ]:
def save_training_info(models, cv_scores):
    """
    保存训练信息
    """
    import json
    
    training_info = {
        'model_type': 'XGBoost',
        'n_folds': CONFIG.N_FOLDS,
        'n_models': len(models),
        'cv_scores': cv_scores,
        'mean_cv_score': float(np.mean(cv_scores)),
        'std_cv_score': float(np.std(cv_scores)),
        'feature_cols': CONFIG.FEATURE_COLS,
        'target_col': CONFIG.TARGET_COL,
        'config': {
            'n_estimators': CONFIG.N_ESTIMATORS,
            'learning_rate': CONFIG.LEARNING_RATE,
            'max_depth': CONFIG.MAX_DEPTH,
            'min_child_weight': CONFIG.MIN_CHILD_WEIGHT,
            'subsample': CONFIG.SUBSAMPLE,
            'colsample_bytree': CONFIG.COLSAMPLE_BYTREE,
            'reg_alpha': CONFIG.REG_ALPHA,
            'reg_lambda': CONFIG.REG_LAMBDA,
        }
    }
    
    info_path = MODEL_OUTPUT_PATH / 'xgb_training_info.json'
    with open(info_path, 'w') as f:
        json.dump(training_info, f, indent=2)
    
    print(f"训练信息已保存到: {info_path}")
    
    return training_info

if 'models' in locals() and models:
    training_info = save_training_info(models, cv_scores)

## 11. 总结

In [ ]:
print("""
=== XGBoost训练总结 ===

1. XGBoost vs LightGBM:
   - XGBoost更稳定，适合生产环境
   - LightGBM训练速度更快
   - 两者性能通常相近

2. 超参数调优建议:
   - learning_rate: 0.01-0.1
   - max_depth: 4-8
   - min_child_weight: 50-200
   - subsample: 0.6-0.9
   - colsample_bytree: 0.6-0.9
   - reg_alpha: 0-2
   - reg_lambda: 0-2

3. 下一步:
   - 尝试神经网络模型
   - 进行模型集成
   - 优化特征工程

4. 推理时的使用:
   - 加载训练好的模型
   - 处理lags数据
   - 进行预测
""")

## 附录：快速推理示例

In [ ]:
def predict_with_xgb_model(test_df, lags_df, model_dir=MODEL_OUTPUT_PATH):
    """
    使用训练好的XGBoost模型进行预测
    """
    model_files = list(model_dir.glob('xgb_model_*.pkl'))
    models = [joblib.load(f) for f in model_files]
    
    # 处理lags
    responder_cols = [f'responder_{i}' for i in range(9)]
    existing_cols = [col for col in responder_cols if col in lags_df.columns]
    
    for col in existing_cols:
        lag_col = f'{col}_lag_1'
        test_df[lag_col] = test_df['symbol_id'].map(lags_df.set_index('symbol_id')[col])
    
    # 填充缺失值
    for col in CONFIG.FEATURE_COLS:
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(0)
    
    X = test_df[CONFIG.FEATURE_COLS]
    predictions = np.mean([model.predict(X) for model in models], axis=0)
    
    return predictions

print("\n推理函数已定义")